[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ach0000/DualScoreMatching/blob/main/tp/TP_entrainement_dual.ipynb)

# Entraîner un réseau d'énergie par score matching dual

On entraîne, avec le code des auteurs, trois réseaux de même architecture (UNet sans biais) : un réseau de score (score matching ordinaire, comme un modèle de diffusion), un réseau d'énergie $U_\theta = \frac12\langle y, s_\theta\rangle$ entraîné avec le seul terme d'espace (score matching « simple ») et le même réseau entraîné par score matching dual. On compare leur coût, leur débruitage, la cohérence de leur normalisation et leur NLL.

**Sur Colab** : `Exécution > Modifier le type d'exécution > GPU T4`, puis exécuter la cellule d'installation.

**Profils** : sur CPU, version jouet : images 16×16 (validation d'ImageNet64 réduite d'un facteur 4), petit UNet, bruits de variance $10^{-5}$ à $10^2$, 8 000 itérations par réseau (environ une heure en tout). Sur GPU, version réaliste : images 64×64 et bruits de $10^{-9}$ à $10^3$ comme les auteurs, UNet plus large, 20 000 itérations (environ une heure et demie sur T4, non vérifié). Les réseaux des auteurs ont été entraînés 10^6 itérations (5 jours de GPU H100).

**Reprise** : les calculs longs sont mis en cache dans `resultats/cache/` ; une exécution interrompue reprend là où elle s'était arrêtée. Supprimer ce dossier pour tout recalculer.

In [ ]:
# Installation (Colab) : à exécuter en premier ; en local, ne fait que rendre le code importable
GITHUB_USER = "ach0000"  # propriétaire du fork
REPO = "DualScoreMatching"
BRANCH = "main"
# seuls ces fichiers du dépôt sont téléchargés (motifs de git sparse-checkout)
FICHIERS = ['/*.py',
            '/networks/',
            '/tp/tp_outils.py']

import os, sys, subprocess
if "google.colab" in sys.modules:
    root = os.path.join("/content", REPO)
    if not os.path.isdir(os.path.join(root, ".git")):
        subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none", "--no-checkout", "-b", BRANCH,
                        "https://github.com/%s/%s.git" % (GITHUB_USER, REPO), root], check=True)
    subprocess.run(["git", "-C", root, "sparse-checkout", "set", "--no-cone", *FICHIERS], check=True)
    subprocess.run(["git", "-C", root, "checkout", BRANCH], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", 'einops', 'PyWavelets'], check=True)
    os.chdir(os.path.join(root, "tp"))
else:
    racine = os.getcwd()
    while not os.path.exists(os.path.join(racine, "main.py")) and os.path.dirname(racine) != racine:
        racine = os.path.dirname(racine)
    os.chdir(os.path.join(racine, "tp"))
sys.path.insert(0, os.getcwd())
print("Répertoire courant :", os.getcwd())

## 0. Réglages et données

Images d'entraînement : les 45 000 premières images de validation d'ImageNet64 (en niveaux de gris) ; images de
test : les 5 000 dernières. Les réseaux des auteurs, eux, ont été entraînés sur les 1,28 million d'images
d'entraînement.

In [ ]:
%matplotlib inline
import math, time
import numpy as np
import torch
import matplotlib.pyplot as plt
import tp_outils as T
from data import DatasetInfo
T.nommer_figures("TP_entrainement_dual")

GPU = torch.cuda.is_available()
if GPU:   # version réaliste : images et plage de bruit des auteurs
    FACTEUR, LARGEUR, ECHELLES, COUCHES, N_ITERATIONS, TAILLE_LOT = 1, 32, 3, 2, 20_000, 64
    T_MIN_E, T_MAX_E, LR, N_DIVISIONS = 1e-9, 1e3, 5e-4, 4
else:     # version jouet
    FACTEUR, LARGEUR, ECHELLES, COUCHES, N_ITERATIONS, TAILLE_LOT = 4, 16, 2, 2, 8000, 64
    T_MIN_E, T_MAX_E, LR, N_DIVISIONS = 1e-5, 1e2, 1e-3, 3
GRAINE = 0
print("calcul sur", T.DEVICE)

tout = T.sous_echantillonner(T.charger_imagenet64_val(), FACTEUR)
X_app, X_test = tout[:45_000], tout[45_000:]
K = X_app.shape[-1]
info = DatasetInfo(spatial_size=K, num_channels=1, mean=X_app.mean().item(), variance=X_app.var().item())
d = info.dimension
print("images %dx%d, d = %d ; apprentissage %d, test %d" % (K, K, d, len(X_app), len(X_test)))
T.afficher_images(X_test[:10], titre="images de test")

## 1. Un modèle de référence : la gaussienne

Spectre de la covariance des images d'apprentissage. Il sert deux fois : NLL du modèle gaussien (le meilleur modèle
linéaire), et entropie de la gaussienne de même covariance que $y = x + \sqrt t z$, qui permet de tester la
normalisation à plusieurs niveaux de bruit.

In [ ]:
Xa = X_app.flatten(1).double()
mu = Xa.mean(0)
Sigma = (Xa - mu).T @ (Xa - mu) / len(Xa) + torch.eye(d, dtype=torch.float64) / (12 * 256 ** 2)
lam = torch.linalg.eigvalsh(Sigma)
L = torch.linalg.cholesky(Sigma)
w = torch.linalg.solve_triangular(L, (X_test.flatten(1).double() - mu).T, upper=False)
e_gauss = 0.5 * (w ** 2).sum(0) + torch.log(torch.diagonal(L)).sum() + d / 2 * math.log(2 * math.pi)
print("modèle gaussien : NLL de test %.3f bits/dim" % T.nll_discrete_bits(e_gauss, d).mean())

## 2. Entraînement des trois réseaux

`T.entrainer_reseau` reprend la boucle de `main.py` : $\log t$ uniforme dans $[t_{\min}, t_{\max}]$ ($[10^{-9}, 10^3]$
chez les auteurs et dans la version GPU ; $[10^{-5}, 10^2]$ dans la version jouet, où les niveaux de bruit très faibles
n'apprennent rien en si peu d'itérations), perte
$\frac1d\|\sqrt t\nabla_yU_\theta - z\|^2$ (terme d'espace), plus, pour le score matching dual,
$\big(\frac td\partial_tU_\theta - \frac12(1 - \|z\|^2/d)\big)^2$ (terme de temps). Pour le réseau de score, le
débruiteur est directement $y - t\, s_\theta(y,t)$ ; pour les réseaux d'énergie, $y - t\nabla_yU_\theta(y,t)$, ce qui
demande une rétropropagation dans la passe avant, et une seconde pour l'apprentissage.

In [ ]:
reseaux, historiques = {}, {}
for nom, type_reseau, perte in [("score", "score", "dsm"), ("énergie, simple", "energie", "dsm"),
                                ("énergie, dual", "energie", "dual")]:
    print(nom)
    m = T.construire_reseau(info, type_reseau, base_width=LARGEUR, num_scales=ECHELLES, n_couches=COUCHES, graine=GRAINE)
    historiques[nom] = T.entrainer_reseau(m, X_app, perte, n_iterations=N_ITERATIONS, taille_lot=TAILLE_LOT, lr=LR,
                                          t_min=T_MIN_E, t_max=T_MAX_E, n_divisions=N_DIVISIONS, graine=GRAINE,
                                          cache=type_reseau + "_" + perte, afficher_tous=1000)
    reseaux[nom] = m
print("paramètres : %.2f millions" % (m.nb_parametres / 1e6))

fig, axes = plt.subplots(1, 2, figsize=(10, 3.2))
lisse = lambda v: np.convolve(v, np.ones(50) / 50, mode="valid")
for nom, h in historiques.items():
    axes[0].semilogy(lisse(h[:, 0]), label=nom)
    if nom != "score":
        axes[1].semilogy(lisse(h[:, 1]), label=nom)
axes[0].set_title("terme d'espace"); axes[1].set_title("terme de temps")
for ax in axes:
    ax.set_xlabel("itération"); ax.legend(fontsize=8)
T.montrer("pertes")
duree = {nom: np.median(h[10:, 2]) for nom, h in historiques.items()}
for nom, v in duree.items():
    print("%-16s : %.3f s par itération (x %.1f par rapport au réseau de score)" % (nom, v, v / duree["score"]))

## 3. Débruitage

PSNR de sortie en fonction du PSNR d'entrée $-10\log_{10}t$, sur 256 images de test.

In [ ]:
psnr_entree = [60, 45, 30, 15, 0, -15, -30]
Xt = X_test[:256]
print("PSNR d'entrée   " + " ".join("%7d" % p for p in psnr_entree))
courbes = {}
for nom, m in reseaux.items():
    T.fixer_graine(GRAINE)
    ligne = []
    for p in psnr_entree:
        t = 10 ** (-p / 10)
        Xh = m.debruiter(Xt + math.sqrt(t) * torch.randn_like(Xt), t)
        ligne.append((-10 * torch.log10(((Xh - Xt) ** 2).mean())).item())
    courbes[nom] = ligne
    print("%-16s" % nom + " ".join("%7.2f" % v for v in ligne))
plt.figure(figsize=(5, 3.5))
for nom, v in courbes.items():
    plt.plot(psnr_entree, v, "o-", label=nom)
plt.plot(psnr_entree, psnr_entree, "k:", lw=1, label="identité")
plt.xlabel("PSNR d'entrée (dB)"); plt.ylabel("PSNR de sortie (dB)"); plt.legend(fontsize=8); plt.grid(alpha=0.3)
T.montrer("debruitage")

## 4. Normalisation et NLL

Pour chaque réseau d'énergie : constante $C(t) = h(t) - \mathbb E_y[U_\theta(y,t)]$, où $h(t)$ est l'entropie de la
gaussienne isotrope ($\frac d2\log 2\pi e t$, équation (9)) ou de la gaussienne de même covariance que $y$. Si le
modèle conserve la masse au cours de la diffusion, $C(t)$ ne dépend pas de $t$ pour $t$ grand. On fixe ensuite
$C = C(t_{\max})$ et l'on calcule la NLL de test.

In [ ]:
for nom in ["énergie, simple", "énergie, dual"]:
    m = reseaux[nom]
    print(nom)
    for t in [T_MAX_E, T_MAX_E / 10, T_MAX_E / 100, T_MAX_E / 1000]:
        Cc, e = T.constante_normalisation(m, X_test[:2000], t_max=t)
        h_cov = 0.5 * torch.log(2 * math.pi * math.e * (lam + t)).sum().item()
        print("  t = %6.0e : C isotrope %10.1f ; C (gaussienne de même covariance) %10.1f" % (t, Cc, h_cov - e.mean().item()))
    Cc, _ = T.constante_normalisation(m, X_test[:2000], t_max=T_MAX_E)
    m.fixer_constante(Cc)
    nll_test = T.nll_discrete_bits(m.energie(X_test), d).mean().item()
    nll_app = T.nll_discrete_bits(m.energie(X_app[:5000]), d).mean().item()
    print("  NLL : apprentissage %.3f bits/dim, test %.3f bits/dim" % (nll_app, nll_test))

e_s = reseaux["énergie, simple"].energie(X_test[:2000])
e_d = reseaux["énergie, dual"].energie(X_test[:2000])
plt.figure(figsize=(4.2, 4))
plt.scatter(T.nats_vers_dB(e_d, d), T.nats_vers_dB(e_s, d), s=3)
lim = [T.nats_vers_dB(e_d, d).min().item(), T.nats_vers_dB(e_d, d).max().item()]
plt.plot(lim, lim, "k:", lw=1)
plt.xlabel("log p, score matching dual (dB/dim)"); plt.ylabel("log p, score matching simple (dB/dim)")
T.montrer("simple_contre_dual")
print("corrélation des log p simple / dual : %.3f ; écart moyen %.2f dB/dim"
      % (np.corrcoef(e_s, e_d)[0, 1], (T.nats_vers_dB(e_s, d) - T.nats_vers_dB(e_d, d)).mean()))

## 5. Le score de temps des deux réseaux d'énergie

Le score matching simple n'impose rien au score de temps $\partial_tU_\theta$ : il n'est déterminé que par la
façon dont le réseau a interpolé entre les niveaux de bruit. On teste, sur des images de test bruitées, l'équation de
diffusion $\partial_tU = \frac12\Delta U - \frac12\|\nabla U\|^2$ (laplacien par la méthode de Hutchinson),
que vérifie l'énergie exacte.

In [ ]:
xh = X_test[:8].to(T.DEVICE)
for nom in ["énergie, simple", "énergie, dual"]:
    m = reseaux[nom]
    print(nom)
    for t in [1e-4, 1e-2, 1.0, 1e2]:
        T.fixer_graine(GRAINE)
        y = xh + math.sqrt(t) * torch.randn_like(xh)
        U, gy, gt, lap, err = T.laplacien_hutchinson(m, y, t, n_sondes=8)
        gauche = (t * gt / d).mean().item()
        droite = (t / 2 * (lap - (gy ** 2).flatten(1).sum(1)) / d).mean().item()
        print("  t = %6.0e : t dU/dt / d = %8.4f ; t (lap U - |grad U|^2) / 2d = %8.4f (± %.4f)"
              % (t, gauche, droite, (t / 2 * err / d).mean().item()))